# **Project: Cambodian Tourism Chatbot (RNN)**
---


### **1. Project Overview**

> In this project, you will build a Natural Language Processing (NLP) model using a **Simple Recurrent Neural Network (RNN)**. The goal is to create a chatbot that can answer common questions about tourism in Cambodia, covering destinations, food, and travel logistics
---


### **2. Environment Setup**

Import all necessary libraries for data manipulation, visualization, and deep learning.


In [ ]:
# Import pandas, numpy, matplotlib, and tensorflow
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

### **3. Data Acquisition & Exploration**
#### **3.1 Load Dataset**
**Instructions**: Load the `cambodia_tourism_dataset_large.csv` file and display the first 10 rows.


In [ ]:
# TODO: Load the CSV file
# TODO: Display the head of the dataframe

#### **3.2 Dataset Analysis**
**Instructions**: Perform basic EDA. Calculate the total number of samples and the maximum length of the questions and answers.


In [ ]:
# TODO: Print total number of rows
# TODO: Find the maximum length (word count) of the 'question' and 'answer' columns

### **4. Data Preprocessing**
#### **4.1 Tokenization**
Convert text into numerical format. Fit a tokenizer on both questions and answers to build a shared vocabulary.


In [ ]:
# Correct column names from generated CSV
inputs = df["question"].astype(str).tolist()
responses = df["answer"].astype(str).tolist()

In [ ]:
# Tokenization
## Initialize the Tokenizer
tokenizer = Tokenizer(filters='') # Keep it simple for RNN
tokenizer.fit_on_texts(inputs + responses) # Fit on text
vocab_size = len(tokenizer.word_index) + 1 

X_seq = tokenizer.texts_to_sequences(inputs)
y_seq = tokenizer.texts_to_sequences(responses)

In [ ]:
# 1. See how many unique words were found
print(f"Total words in vocabulary: {vocab_size}")

# 2. Look at the first 10 words in the dictionary
print("Word Index Sample:", dict(list(tokenizer.word_index.items())[:10]))

# 3. Verify the conversion
sample_text = inputs[0]
sample_seq = X_seq[0]
print(f"\nOriginal: {sample_text}")
print(f"Numerical: {sample_seq}")

#### Explanation
Since neural networks like a `SimpleRNN` cannot "read" text, we must convert words into unique integers.

**1. `Tokenizer(filters='')`**
By default, a tokenizer removes punctuation (like `?`, `!`, and `.`). By setting `filters=''`, we tell the model to keep the text exactly as it is. This is important if you want the chatbot to recognize the difference between a statement and a question.

**2. `tokenizer.fit_on_texts(inputs + responses)`**
This is the "learning" phase for the tokenizer. It scans every word in your dataset and builds a dictionary (a **Word Index**). Every unique word is assigned a specific number. 
*   *Example:* "Angkor" becomes `1`, "Wat" becomes `2`, "Siem" becomes `3`, and so on.

**3. `vocab_size = len(tokenizer.word_index) + 1`**
This calculates the total number of unique words the model knows. We add `+1` because the computer reserves the number **`0`** for a special task called "Padding" (filling empty space in short sentences).

**4. `texts_to_sequences`**
This is the final transformation. It takes your actual sentences and replaces the words with the numbers from the dictionary.

---

**Before (Text):**
> `"where is angkor wat"`

**The Word Index (Dictionary):**
> `{'where': 1, 'is': 2, 'angkor': 3, 'wat': 4}`

**After (`X_seq`):**
> `[1, 2, 3, 4]`

---


#### **4.2 Sequencing and Padding**
**Instructions**: Convert the text strings into sequences of integers and pad them so they all have the same length. Explain why padding is necessary for RNNs.


In [ ]:
# Padding
# Calculate max_len based on 95th percentile or max to keep sequences uniform
max_len = max(max(len(seq) for seq in X_seq), max(len(seq) for seq in y_seq))

# TODO: Pad the sequences to ensure uniform length for model input


#### **4.3 Train-Test Split**
**Instructions**: Split the data into 80% training and 20% testing sets to evaluate your model's performance on unseen data.


In [ ]:
# TODO: Split X and y using train_test_split
X_train, X_test, y_train, y_test = ...

In [ ]:
# 5. One-hot encoding for the targets
# Note: With large vocab_size, this uses a lot of RAM. 
y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=vocab_size)
y_test_oh = tf.keras.utils.to_categorical(y_test, num_classes=vocab_size)

This step, known as **One-Hot Encoding**, is where we transform our target word IDs into a format that the model can use to calculate "how wrong" its guess was during training.

**1. What is One-Hot Encoding?**
In the previous step, we turned words into integers (e.g., "Angkor" = 3). However, we cannot simply use the number `3` as a final answer for the model to predict. If the model guesses `2` instead of `3`, the math would suggest it was "close." In language, word #2 and word #3 are completely different concepts, not "close" numbers.

**One-Hot Encoding** fixes this by turning each integer into a **binary vector**. Every position in the vector is `0`, except for the specific index of the word, which is `1`.

**Example:**
If our vocabulary is only 5 words: `[apple, banana, angkor, wat, siem]`
*   The word **"angkor"** (index 2) becomes: `[0, 0, 1, 0, 0]`
*   The word **"apple"** (index 0) becomes: `[1, 0, 0, 0, 0]`

**2. Why use `tf.keras.utils.to_categorical`?**
This function automates that transformation. It takes your list of integers (`y_train`) and creates a massive table (matrix) of `0`s and `1`s. This allows the final layer of your RNN (the **Softmax** layer) to output a probability for every single word in your dictionary.

In [ ]:
# Look at the first word of the first response as an integer
print("Integer format:", y_train[0][0])

# Look at that same word after One-Hot Encoding
print("One-Hot format (first 10 slots):", y_train_oh[0][0][:10])

# Verify the shape of the data
print("Shape of One-Hot data:", y_train_oh.shape)

### **5. Model Architecture Design**
#### **5.1 Design Strategy**
**Instructions**: Outline and define your model architecture. You are required to use an **Embedding Layer** and a **SimpleRNN**. 

In [ ]:
# TODO: Define the Sequential model
# TODO: Add Embedding layer (with mask_zero=True)
# TODO: Add SimpleRNN layer
# TODO: Add Dense layer with Softmax activation

#### **5.2 Compilation**
**Instructions**: Compile the model

In [ ]:
# TODO: Compile the model

### **6. Training and Evaluation**
#### **6.1 Model Training**
**Instructions**: Train the model for at least 50-100 epochs. Use the test set as validation data.


In [ ]:
# TODO: Run model.fit() and save the history

#### **6.2 Performance Visualization**
**Instructions**: Plot the training and validation loss over time. Identify if the model is overfitting.


In [ ]:
# TODO: Create a line chart for loss and accuracy

### **7. Testing the Chatbot**
#### **7.1 Inference Test**
**Instructions**: Write a prediction. Able to do string input, processes it, and returns the model's predicted response.


In [ ]:
# TODO: Write a code to do prediction answers for new questions
# TODO: Test with 3 different questions about Cambodia Tourism and print the predicted answers

### **8. Conclusion & Reflection**
**Instructions**: Summarize your findings. What were the challenges with the SimpleRNN? How would you improve this chatbot?


#### Write your reflection here
-
-
-
-